In [ ]:
import pandas as pd
df_results = pd.read_excel('./data/sampling_logical_fallacy_data.xlsx')
df_gt = pd.read_excel('./data/sampling_ground_truth.xlsx') 

In [4]:
df_results.groupby('model_name').size()

model_name
anthropic/claude-sonnet-4.5           107
deepseek/deepseek-chat-v3.1           107
deepseek/deepseek-r1                  107
gemini-flash-non                      107
gemini-flash-thinking                 107
gemini-pro-non                        107
gemini-pro-thinking                   107
moonshotai/kimi-k2-thinking           107
openai/gpt-4o-2024-11-20              107
openai/gpt-5                          107
openai/gpt-oss-120b                   107
qwen/qwen3-235b-a22b-thinking-2507    107
qwen_qwen3-235b-a22b-2507             107
x-ai/grok-4-fast-thinking             107
x-ai_grok-4-fast                      107
dtype: int64

In [ ]:
import pandas as pd

# 1. Define the list of 13 Canonical Fallacies
# Standardized format: lowercase, underscores replaced with spaces
canonical_fallacies = [
    'ad hominem', 'ad populum', 'appeal to emotion', 'circular reasoning', 
    'equivocation', 'fallacy of credibility', 'fallacy of extension', 
    'fallacy of logic', 'fallacy of relevance', 'false causality', 
    'false dilemma', 'faulty generalization', 'intentional'
]

# 2. Define cleaning function (for reusability)
def clean_label(text):
    if pd.isna(text):
        return None
    # Convert to string -> replace underscores with spaces -> lowercase -> trim whitespace
    return str(text).replace('_', ' ').lower().strip()

# 3. Clean df_gt (Ground Truth)
def clean_gt_list(gt_str):
    if pd.isna(gt_str):
        return set()
    return {clean_label(item) for item in str(gt_str).split(',')}

df_gt['cleaned_gt_set'] = df_gt['ground_truth'].apply(clean_gt_list)

# 4. Clean df_results (Model Results)
# --- Clean fallacy_option 1, 2, 3 (keep original structure) ---
option_cols = ['fallacy_option1', 'fallacy_option2', 'fallacy_option3']
for col in option_cols:
    df_results[col] = df_results[col].apply(clean_label)

# --- 5. Create "Exploded Predictions" table and implement "no_fallacy" logic ---

# A. Identify questions with "No Answer" (all three options are null)
# Identified based on model_name and question_id
df_results['is_no_answer'] = df_results[option_cols].isnull().all(axis=1)

# B. Process questions with answers
opt1 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option1', 'verification_status1']].rename(
    columns={'fallacy_option1': 'prediction', 'verification_status1': 'status'})
opt2 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option2', 'verification_status2']].rename(
    columns={'fallacy_option2': 'prediction', 'verification_status2': 'status'})
opt3 = df_results[~df_results['is_no_answer']][['model_name', 'question_id', 'fallacy_option3', 'verification_status3']].rename(
    columns={'fallacy_option3': 'prediction', 'verification_status3': 'status'})

# Concatenate answered rows and remove NaNs (e.g., if a question only has 2 answers, the 3rd NaN is dropped)
df_answered = pd.concat([opt1, opt2, opt3], ignore_index=True).dropna(subset=['prediction'])

# C. Handle "Completely Unanswered" questions (assign "no_fallacy")
df_no_fallacy = df_results[df_results['is_no_answer']][['model_name', 'question_id']].copy()
df_no_fallacy['prediction'] = 'no_fallacy'
df_no_fallacy['status'] = 'no_pass'  # Default status for no answer is "no_pass"

# D. Merge both to form the global denominator table
df_preds_global = pd.concat([df_answered, df_no_fallacy], ignore_index=True)

# --- 6. Merge cleaned Ground Truth into the exploded table ---
df_preds_global = pd.merge(
    df_preds_global, 
    df_gt[['question_id', 'cleaned_gt_set']], 
    on='question_id', 
    how='left'
)

print("Global data preparation complete (including no_fallacy logic)!")
print(f"Total processed predictions (Total Denominator): {len(df_preds_global)}")
# Checking the count of unanswered questions now occupying rows
print(f"Number of 'no_fallacy' entries: {len(df_preds_global[df_preds_global['prediction'] == 'no_fallacy'])}")

Global data preparation complete (including no_fallacy logic)!
Total processed predictions (Total Denominator): 3621
Number of 'no_fallacy' entries: 90


In [10]:
# Detect cases where questions are labeled as no_fallacy
import pandas as pd
import numpy as np

# 1. Define option-related columns to check
# Based on DataFrame structure, from 'fallacy_option1' to 'explanation3'
option_columns = [
    'fallacy_option1', 'verification_status1', 'fallacy_option2', 'verification_status2',
    'fallacy_option3', 'verification_status3'
]

# 2. Filter rows that match "Empty Case 1"

# Condition a: all option-related columns are empty (NaN / None)
# .isna() checks for empty values, .all(axis=1) ensures all columns are empty
condition_A = df_results[option_columns].isna().all(axis=1)
df_empty_case1 = df_results[condition_A].copy()


# 3. Group results by model_name

# Count how many Empty Case 1 questions per model
empty_case1_counts = (
    df_empty_case1
    .groupby('model_name')
    .size()
    .reset_index(name='Empty_Case1_Count')
)

# Get question_id list for each model
empty_case1_qids = (
    df_empty_case1
    .groupby('model_name')['question_id']
    .apply(list)
    .reset_index(name='Empty_Case1_QIDs')
)

# Merge results
df_diagnostic = pd.merge(
    empty_case1_counts,
    empty_case1_qids,
    on='model_name',
    how='outer'
)

# Format output so models without this case show 0
df_diagnostic = df_diagnostic.fillna(0)
df_diagnostic['Empty_Case1_Count'] = df_diagnostic['Empty_Case1_Count'].astype(int)


# 4. Print report
print("--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---")
print(df_diagnostic)

--- Model Diagnostic Report: Empty Case 1 (Context/GT present, but all options empty) ---
                            model_name  Empty_Case1_Count  \
0          anthropic/claude-sonnet-4.5                  6   
1          deepseek/deepseek-chat-v3.1                 11   
2                 deepseek/deepseek-r1                 12   
3                     gemini-flash-non                  8   
4                gemini-flash-thinking                  1   
5                       gemini-pro-non                  1   
6                  gemini-pro-thinking                  2   
7          moonshotai/kimi-k2-thinking                 10   
8             openai/gpt-4o-2024-11-20                  6   
9                         openai/gpt-5                  4   
10                 openai/gpt-oss-120b                  7   
11  qwen/qwen3-235b-a22b-thinking-2507                  7   
12           qwen_qwen3-235b-a22b-2507                  3   
13           x-ai/grok-4-fast-thinking                 1

In [11]:
#  Define constants
PASS_STATUS = 'lean_pass'

# Logical failure (new core error type)
LOGICAL_FAIL_STATUS = 'lean_pass_with_type_error'

# Technical failure (independent metric)
TECHNICAL_FAIL_STATUS = 'no_pass'


# Total rows per model (denominator)
total_rows = (
    df_results.groupby('model_name')
    .size()
    .reset_index(name='Total_Count')
)

In [13]:
# Compilable-Correct @3 (Label Match + Lean4 Pass)

df_preds_global['is_valid_correct'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] in row['cleaned_gt_set'] and row['status'] == 'lean_pass') else 0, 
    axis=1
)

# Summary of correct hits per model
valid_correct_summary = df_preds_global.groupby('model_name')['is_valid_correct'].sum().reset_index()
print("Compilable-Correct Hits per Model (Label Matched & Verification Passed):")
print(valid_correct_summary)

Compilable-Correct Hits per Model (Label Matched & Verification Passed):
                            model_name  is_valid_correct
0          anthropic/claude-sonnet-4.5                52
1          deepseek/deepseek-chat-v3.1                39
2                 deepseek/deepseek-r1                45
3                     gemini-flash-non                43
4                gemini-flash-thinking                49
5                       gemini-pro-non                49
6                  gemini-pro-thinking                54
7          moonshotai/kimi-k2-thinking                51
8             openai/gpt-4o-2024-11-20                54
9                         openai/gpt-5                52
10                 openai/gpt-oss-120b                48
11  qwen/qwen3-235b-a22b-thinking-2507                48
12           qwen_qwen3-235b-a22b-2507                58
13           x-ai/grok-4-fast-thinking                45
14                    x-ai_grok-4-fast                50


In [14]:
# Compilable-Alternative @3 (Label Mismatch + Lean 4 Pass)
df_preds_global['is_valid_alternative'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] not in row['cleaned_gt_set'] and 
                      row['status'] == 'lean_pass' ) else 0, 
                    # and row['prediction'] in canonical_fallacies 
    axis=1
)

# Preview the number of alternative correct hits per model
valid_alt_summary = df_preds_global.groupby('model_name')['is_valid_alternative'].sum().reset_index()
print("Compilable-Alternative Hits per Model:")
print(valid_alt_summary)

Compilable-Alternative Hits per Model:
                            model_name  is_valid_alternative
0          anthropic/claude-sonnet-4.5                   161
1          deepseek/deepseek-chat-v3.1                   135
2                 deepseek/deepseek-r1                   140
3                     gemini-flash-non                   134
4                gemini-flash-thinking                   156
5                       gemini-pro-non                   177
6                  gemini-pro-thinking                   223
7          moonshotai/kimi-k2-thinking                   209
8             openai/gpt-4o-2024-11-20                   174
9                         openai/gpt-5                   231
10                 openai/gpt-oss-120b                   179
11  qwen/qwen3-235b-a22b-thinking-2507                   171
12           qwen_qwen3-235b-a22b-2507                   187
13           x-ai/grok-4-fast-thinking                   150
14                    x-ai_grok-4-fast        

In [15]:
# Uncompilable-Correct @3 (Label Match + Lean 4 Fail)
df_preds_global['is_invalid_correct'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] in row['cleaned_gt_set'] and row['status'] != 'lean_pass') else 0, 
    axis=1
)

# Statistics for Uncompilable-Correct counts per model
invalid_correct_summary = df_preds_global.groupby('model_name')['is_invalid_correct'].sum().reset_index()

print("Uncompilable-Correct Hits per Model:")
print(invalid_correct_summary)

Uncompilable-Correct Hits per Model:
                            model_name  is_invalid_correct
0          anthropic/claude-sonnet-4.5                   2
1          deepseek/deepseek-chat-v3.1                   2
2                 deepseek/deepseek-r1                   1
3                     gemini-flash-non                   2
4                gemini-flash-thinking                   6
5                       gemini-pro-non                   5
6                  gemini-pro-thinking                   3
7          moonshotai/kimi-k2-thinking                   4
8             openai/gpt-4o-2024-11-20                   0
9                         openai/gpt-5                   1
10                 openai/gpt-oss-120b                   0
11  qwen/qwen3-235b-a22b-thinking-2507                   0
12           qwen_qwen3-235b-a22b-2507                   3
13           x-ai/grok-4-fast-thinking                   1
14                    x-ai_grok-4-fast                   3


In [16]:
# --- Cell: Calculate NF (No Fallacy) ---
# Logic: Model chose not to answer (already labeled as 'no_fallacy' in Cell 0)

df_preds_global['is_nf'] = (df_preds_global['prediction'] == 'no_fallacy').astype(int)

# Statistics for NF counts per model
nf_summary = df_preds_global.groupby('model_name')['is_nf'].sum().reset_index()
print("NF Counts per Model (No Answer):")
print(nf_summary)

NF Counts per Model (No Answer):
                            model_name  is_nf
0          anthropic/claude-sonnet-4.5      6
1          deepseek/deepseek-chat-v3.1     11
2                 deepseek/deepseek-r1     12
3                     gemini-flash-non      8
4                gemini-flash-thinking      1
5                       gemini-pro-non      1
6                  gemini-pro-thinking      2
7          moonshotai/kimi-k2-thinking     10
8             openai/gpt-4o-2024-11-20      6
9                         openai/gpt-5      4
10                 openai/gpt-oss-120b      7
11  qwen/qwen3-235b-a22b-thinking-2507      7
12           qwen_qwen3-235b-a22b-2507      3
13           x-ai/grok-4-fast-thinking     11
14                    x-ai_grok-4-fast      1


In [17]:
# --- Cell: Calculate SF (Syntax Failure) ---
# Logic: A prediction was provided but the status is 'no_pass' (Technical/Syntax Failure)

df_preds_global['is_sf'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] != 'no_fallacy' and row['status'] == 'no_pass') else 0, 
    axis=1
)

# Statistics for SF counts per model
sf_summary = df_preds_global.groupby('model_name')['is_sf'].sum().reset_index()
print("SF Counts per Model (Syntax/Technical Failure):")
print(sf_summary)

SF Counts per Model (Syntax/Technical Failure):
                            model_name  is_sf
0          anthropic/claude-sonnet-4.5      1
1          deepseek/deepseek-chat-v3.1      5
2                 deepseek/deepseek-r1      4
3                     gemini-flash-non      2
4                gemini-flash-thinking      3
5                       gemini-pro-non      8
6                  gemini-pro-thinking     11
7          moonshotai/kimi-k2-thinking      3
8             openai/gpt-4o-2024-11-20      0
9                         openai/gpt-5      0
10                 openai/gpt-oss-120b      0
11  qwen/qwen3-235b-a22b-thinking-2507      6
12           qwen_qwen3-235b-a22b-2507      3
13           x-ai/grok-4-fast-thinking      3
14                    x-ai_grok-4-fast      2


In [ ]:
# --- Cell: Calculate VF (Verification Failure) ---
# Logic: Not in Ground Truth (GT) + Status indicates a logical/verification error 
# (Specifically 'lean_pass_with_type_error' in this context)

df_preds_global['is_vf'] = df_preds_global.apply(
    lambda row: 1 if (row['prediction'] not in row['cleaned_gt_set'] and 
                    #   row['prediction'] in canonical_fallacies and
                      row['status'] == 'lean_pass_with_type_error') else 0, 
    axis=1
)

# Statistics for VF counts per model
vf_summary = df_preds_global.groupby('model_name')['is_vf'].sum().reset_index()
print("VF Counts per Model (Verification/Logical Error):")
print(vf_summary)

VF Counts per Model (Verification/Logical Error):
                            model_name  is_vf
0          anthropic/claude-sonnet-4.5      5
1          deepseek/deepseek-chat-v3.1      2
2                 deepseek/deepseek-r1      0
3                     gemini-flash-non      6
4                gemini-flash-thinking     14
5                       gemini-pro-non      8
6                  gemini-pro-thinking     10
7          moonshotai/kimi-k2-thinking      2
8             openai/gpt-4o-2024-11-20      2
9                         openai/gpt-5      2
10                 openai/gpt-oss-120b      0
11  qwen/qwen3-235b-a22b-thinking-2507      5
12           qwen_qwen3-235b-a22b-2507      5
13           x-ai/grok-4-fast-thinking      5
14                    x-ai_grok-4-fast      5


In [19]:
# Uncompilable-Incorrect @3 (Label Mismatch + Lean 4 Fail)
# --- Cell: Total Invalid-Incorrect Summary ---

# Sum the three indicators
df_preds_global['is_invalid_incorrect'] = df_preds_global['is_nf'] + df_preds_global['is_sf'] + df_preds_global['is_vf']

# Aggregate the report
invalid_incorrect_report = df_preds_global.groupby('model_name').agg(
    NF_Count=('is_nf', 'sum'),
    SF_Count=('is_sf', 'sum'),
    VF_Count=('is_vf', 'sum'),
    Total_Invalid_Incorrect=('is_invalid_incorrect', 'sum')
).reset_index()

print("--- Invalid-Incorrect Summary Report (NF + SF + VF) ---")
print(invalid_incorrect_report)

--- Invalid-Incorrect Summary Report (NF + SF + VF) ---
                            model_name  NF_Count  SF_Count  VF_Count  \
0          anthropic/claude-sonnet-4.5         6         1         5   
1          deepseek/deepseek-chat-v3.1        11         5         2   
2                 deepseek/deepseek-r1        12         4         0   
3                     gemini-flash-non         8         2         6   
4                gemini-flash-thinking         1         3        14   
5                       gemini-pro-non         1         8         8   
6                  gemini-pro-thinking         2        11        10   
7          moonshotai/kimi-k2-thinking        10         3         2   
8             openai/gpt-4o-2024-11-20         6         0         2   
9                         openai/gpt-5         4         0         2   
10                 openai/gpt-oss-120b         7         0         0   
11  qwen/qwen3-235b-a22b-thinking-2507         7         6         5   
12      

In [21]:
# --- Cell: Detailed Uncompilable-Incorrect Percentage Summary ---

# 1. Aggregate counts for NF, SF, VF, and total Uncompilable-Incorrect by model
detailed_invalid_stats = df_preds_global.groupby('model_name').agg(
    NF_Count=('is_nf', 'sum'),
    SF_Count=('is_sf', 'sum'),
    VF_Count=('is_vf', 'sum'),
    Invalid_Incorrect_Count=('is_invalid_incorrect', 'sum'),
    Total_Count=('question_id', 'count') # Total denominator (including no_fallacy)
).reset_index()

# 2. Calculate percentages for each sub-category relative to the total
# Formula: (Count / Total_Count) * 100
detailed_invalid_stats['NF_%'] = (detailed_invalid_stats['NF_Count'] / detailed_invalid_stats['Total_Count'] * 100).round(2)
detailed_invalid_stats['SF_%'] = (detailed_invalid_stats['SF_Count'] / detailed_invalid_stats['Total_Count'] * 100).round(2)
detailed_invalid_stats['VF_%'] = (detailed_invalid_stats['VF_Count'] / detailed_invalid_stats['Total_Count'] * 100).round(2)
detailed_invalid_stats['Total_Invalid_Incorrect_%'] = (detailed_invalid_stats['Invalid_Incorrect_Count'] / detailed_invalid_stats['Total_Count'] * 100).round(2)

# 3. Verify Checksum: Ensure NF% + SF% + VF% equals Total_Invalid_Incorrect%
detailed_invalid_stats['Sub_Checksum_%'] = (
    detailed_invalid_stats['NF_%'] + 
    detailed_invalid_stats['SF_%'] + 
    detailed_invalid_stats['VF_%']
).round(2)

# 4. Preview report content
print("--- Uncompilable-Incorrect Detailed Percentage Report ---")
display_cols = ['model_name', 'Total_Count', 'NF_%', 'SF_%', 'VF_%', 'Total_Invalid_Incorrect_%', 'Sub_Checksum_%']
print(detailed_invalid_stats[display_cols])

# 5. Export to Excel file
output_file = 'invalid_incorrect_breakdown_report.xlsx'
detailed_invalid_stats.to_excel(output_file, index=False)

print(f"\n✅ Detailed percentage report exported to: {output_file}")

--- Uncompilable-Incorrect Detailed Percentage Report ---
                            model_name  Total_Count  NF_%  SF_%  VF_%  \
0          anthropic/claude-sonnet-4.5          227  2.64  0.44  2.20   
1          deepseek/deepseek-chat-v3.1          193  5.70  2.59  1.04   
2                 deepseek/deepseek-r1          201  5.97  1.99  0.00   
3                     gemini-flash-non          195  4.10  1.03  3.08   
4                gemini-flash-thinking          229  0.44  1.31  6.11   
5                       gemini-pro-non          248  0.40  3.23  3.23   
6                  gemini-pro-thinking          302  0.66  3.64  3.31   
7          moonshotai/kimi-k2-thinking          276  3.62  1.09  0.72   
8             openai/gpt-4o-2024-11-20          236  2.54  0.00  0.85   
9                         openai/gpt-5          290  1.38  0.00  0.69   
10                 openai/gpt-oss-120b          234  2.99  0.00  0.00   
11  qwen/qwen3-235b-a22b-thinking-2507          237  2.95  2.53  2

## Percentage Result

In [22]:
# --- Cell: Performance Percentage Summary ---

# 1. Aggregate counts for the four core metrics by model_name
final_performance = df_preds_global.groupby('model_name').agg(
    Valid_Correct=('is_valid_correct', 'sum'),
    Valid_Alternative=('is_valid_alternative', 'sum'),
    Invalid_Correct=('is_invalid_correct', 'sum'),
    Invalid_Incorrect=('is_invalid_incorrect', 'sum'),
    Total_Count=('question_id', 'count')  # Global denominator per model (includes no_fallacy)
).reset_index()

# 2. Calculate percentages for each metric
# Formula: (Metric Count / Total_Count) * 100
metrics = ['Valid_Correct', 'Valid_Alternative', 'Invalid_Correct', 'Invalid_Incorrect']
for col in metrics:
    final_performance[f'{col}_%'] = (final_performance[col] / final_performance['Total_Count'] * 100).round(2)

# 3. Preview results
print("--- Model Performance Percentage Statistics ---")
# Select model name, denominator, and the four percentage columns for display
display_cols = ['model_name', 'Total_Count'] + [f'{m}_%' for m in metrics]
print(final_performance[display_cols])

# 4. Export the results
final_performance.to_excel('model_final_percentage_report.xlsx', index=False)
print("\n✅ Percentage report calculation complete and exported to: model_final_percentage_report.xlsx")

--- Model Performance Percentage Statistics ---
                            model_name  Total_Count  Valid_Correct_%  \
0          anthropic/claude-sonnet-4.5          227            22.91   
1          deepseek/deepseek-chat-v3.1          193            20.21   
2                 deepseek/deepseek-r1          201            22.39   
3                     gemini-flash-non          195            22.05   
4                gemini-flash-thinking          229            21.40   
5                       gemini-pro-non          248            19.76   
6                  gemini-pro-thinking          302            17.88   
7          moonshotai/kimi-k2-thinking          276            18.48   
8             openai/gpt-4o-2024-11-20          236            22.88   
9                         openai/gpt-5          290            17.93   
10                 openai/gpt-oss-120b          234            20.51   
11  qwen/qwen3-235b-a22b-thinking-2507          237            20.25   
12           qwe